## STEP 0 — Setup

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Load the Social Media Sentiment dataset
csv_path = Path("Social_Media_Sentiment_Dataset.csv")

# Automatically use the only CSV in the notebook folder if the expected
# filename is not present.
if not csv_path.exists():
    csv_files = list(Path.cwd().glob("*.csv"))
    if len(csv_files) == 1:
        csv_path = csv_files[0]
    else:
        raise FileNotFoundError(
            "Social Media CSV not found. Put the CSV in the same folder "
            "as this notebook and name it 'Social_Media_Sentiment_Dataset.csv'."
        )

df = pd.read_csv(csv_path)

print("Dataset loaded from:", csv_path)
print("Shape:", df.shape)


In [ ]:
df.head()


## STEP 0.1 — Understand the dataset

In [ ]:
df.info()


In [ ]:
df.describe()
# This gives statistical information about the numeric columns.


## STEP 0.2 — Clean the dataset

In [ ]:
# Remove unnecessary index columns
df = df.drop(
    columns=["Unnamed: 0.1", "Unnamed: 0"],
    errors="ignore"
)

# Remove extra spaces from text/category columns
text_columns = df.select_dtypes(
    include="object"
).columns

for column in text_columns:
    df[column] = df[column].str.strip()

# Convert Timestamp into datetime
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    errors="coerce"
)

# Create an Engagement metric
df["Engagement"] = (
    df["Likes"] + df["Retweets"]
)

print("Cleaned dataset shape:", df.shape)


## STEP 0.3 — Check the cleaned columns

In [ ]:
print("Platforms:")
print(df["Platform"].unique())

print("\nNumber of Sentiment Categories:",
      df["Sentiment"].nunique())

print("\nMissing values:")
print(df.isnull().sum())


In [ ]:
df[[
    "Likes",
    "Retweets",
    "Engagement"
]].describe()


## STEP 1 — Boxplot Using Seaborn

### STEP 1.1 — Prepare engagement data

In [ ]:
plot_df = df[
    ["Platform", "Likes", "Retweets", "Engagement"]
].dropna()

plot_df.head()


In [ ]:
plot_df["Platform"].value_counts()


### STEP 1.2 — Create the Boxplot

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=plot_df,
    x="Platform",
    y="Engagement"
)

plt.title(
    "Engagement Distribution by Platform"
)
plt.xlabel("Platform")
plt.ylabel("Engagement")

plt.tight_layout()
plt.show()


### STEP 1.3 — Create a Boxplot for Likes

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=plot_df,
    x="Platform",
    y="Likes"
)

plt.title(
    "Likes Distribution by Platform"
)
plt.xlabel("Platform")
plt.ylabel("Likes")

plt.tight_layout()
plt.show()


## STEP 2 — Histogram + KDE

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="Engagement",
    kde=True,
    bins=15
)

plt.title(
    "Distribution of Social Media Engagement"
)
plt.xlabel("Engagement")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()


## STEP 3 — Correlation Heatmap

In [ ]:
nums_cols = [
    "Retweets",
    "Likes",
    "Year",
    "Month",
    "Day",
    "Hour",
    "Engagement"
]

corr = df[nums_cols].corr()

corr


### STEP 3.1 — Create Heatmap

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="Blues"
)

plt.title(
    "Correlation Heatmap of Social Media Data"
)

plt.tight_layout()
plt.show()


### Correlation values range from:

**-1 to +1**

- **+1** = strong positive relationship
- **0** = little or no linear relationship
- **-1** = strong negative relationship

The heatmap should be interpreted from the values produced by the actual dataset.

### For example, if:

- Likes and Retweets have a positive value, higher Likes tend to be associated with higher Retweets.
- Engagement and Likes have a positive value, Likes contribute strongly to Engagement because Engagement is calculated from Likes + Retweets.

These are interpretations of the calculated relationship, not a claim of causation.

## Step 4 — Violin Plot

In [ ]:
plt.figure(figsize=(8, 5))

sns.violinplot(
    data=df,
    x="Platform",
    y="Engagement"
)

plt.title(
    "Engagement Distribution by Platform"
)
plt.xlabel("Platform")
plt.ylabel("Engagement")

plt.tight_layout()
plt.show()


### Step 4.1 — Create a Violin Plot for Likes

In [ ]:
plt.figure(figsize=(8, 5))

sns.violinplot(
    data=df,
    x="Platform",
    y="Likes"
)

plt.title(
    "Likes Distribution by Platform"
)
plt.xlabel("Platform")
plt.ylabel("Likes")

plt.tight_layout()
plt.show()


## STEP 5 — Platform Analysis

In [ ]:
platforms = df["Platform"].value_counts()

print("Posts by Platform:")
print(platforms)


In [ ]:
plt.figure(figsize=(8, 5))

platforms.plot(
    kind="bar"
)

plt.title(
    "Number of Social Media Posts by Platform"
)
plt.xlabel("Platform")
plt.ylabel("Number of Posts")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
platform_summary = df.groupby(
    "Platform"
)[
    ["Likes", "Retweets", "Engagement"]
].mean().round(2)

print(
    "Average Metrics by Platform:"
)
print(platform_summary)


## STEP 6 — Posts by Year

In [ ]:
posts_by_year = (
    df["Year"]
    .value_counts()
    .sort_index()
)

print(posts_by_year)


In [ ]:
plt.figure(figsize=(8, 5))

posts_by_year.plot(
    kind="line",
    marker="o"
)

plt.title(
    "Number of Social Media Posts by Year"
)
plt.xlabel("Year")
plt.ylabel("Number of Posts")
plt.grid(True)

plt.tight_layout()
plt.show()


## STEP 6.1 — Posts by Month

In [ ]:
posts_by_month = (
    df["Month"]
    .value_counts()
    .sort_index()
)

print(posts_by_month)


In [ ]:
plt.figure(figsize=(9, 5))

posts_by_month.plot(
    kind="line",
    marker="o"
)

plt.title(
    "Number of Social Media Posts by Month"
)
plt.xlabel("Month")
plt.ylabel("Number of Posts")
plt.xticks(range(1, 13))
plt.grid(True)

plt.tight_layout()
plt.show()


## STEP 6.2 — Posts by Hour

In [ ]:
posts_by_hour = (
    df["Hour"]
    .value_counts()
    .sort_index()
)

print(posts_by_hour)


In [ ]:
plt.figure(figsize=(9, 5))

posts_by_hour.plot(
    kind="line",
    marker="o"
)

plt.title(
    "Number of Social Media Posts by Hour"
)
plt.xlabel("Hour")
plt.ylabel("Number of Posts")
plt.grid(True)

plt.tight_layout()
plt.show()


## STEP 7 — Top 15 Sentiments

In [ ]:
top_sentiments = (
    df["Sentiment"]
    .value_counts()
    .head(15)
)

print(top_sentiments)


In [ ]:
plt.figure(figsize=(10, 5))

top_sentiments.plot(
    kind="bar"
)

plt.title(
    "Top 15 Sentiment Categories"
)
plt.xlabel("Sentiment")
plt.ylabel("Number of Posts")
plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()
plt.show()


## STEP 8 — Likes vs Retweets

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="Likes",
    y="Retweets"
)

plt.title(
    "Likes vs Retweets"
)
plt.xlabel("Likes")
plt.ylabel("Retweets")

plt.tight_layout()
plt.show()


## STEP 9 — Conclusion :-

In this lab, I analyzed the Social Media Sentiment dataset using Python, Pandas, Matplotlib, and Seaborn. I used different plots such as boxplots, histograms with KDE, correlation heatmaps, violin plots, platform-wise comparisons, yearly and monthly trend plots, and a Likes-versus-Retweets scatter plot.

The analysis examined the distribution of social-media engagement, differences between platforms, sentiment categories, and time-based activity. The Engagement metric was created from Likes and Retweets to provide an overall measure of interaction.

The visualizations help convert the raw social-media dataset into meaningful analytical insights while keeping the conclusions based on the actual dataset values.

### Dataset note

The supplied Social Media dataset contains 732 rows and 15 columns before cleaning. After removing the two unnecessary index columns and adding Engagement, the working dataset contains 732 rows and 14 columns. The original Sentiment labels are retained because the dataset contains multiple sentiment/emotion categories.